In [34]:
def multisim_err_with_resp_func(rundata, signame, var_name, true_var_name, preselection, selection, truth_def, n_bins=None, x_range=None, bin_edges=None, weightCV="weights", base_weight_Var="weights_no_tune", wname="weightsGenie"):

        """Calculate multisim uncertainties using the response matrix method.

        Each of the given multisim weight columns is expected to contain a list of weights
        for every row that correspond to the weights of the fluctuated "universes". The
        histogram is regenerated for every universe and the covariance matrix is calculated
        from the resulting histograms.

        This particular method, whereby the response matrix is used, is required for xsec analyses.
        Here, the signal and background are treated differently. The covariance for the background
        is calculated as described above. But the covariance for the signal is calculated using the
        response matrix; the variation for a particular universe is calculated by multiplying the
        response matrix for that universe (S^{univ}_{sel} / S^{univ}_{total}) by the total true
        signal CV (central value) prediction.
        
        This function has written to be used with microfit.

        Parameters
        ----------
        rundata : dict
            Dictionary of dataframes for each sample. This is the output from the load_runs()
            function in data_loading.py.
        signame : str
            The name of the MC sample that contains your signal events. This must refer to one of the keys in rundata.
            For my analysis, this is "nue".
        var_name : str
            The name of the reconstructed variable you are calculating the covariance for.
        true_var_name : str
            The name of the truth-level variable corresponding to var_name. This is required, unless var_name and true_var_name
            share the same stem but with either "Reco" or "True" at the front of the string 
            (e.g. var_name = "RecoDeltaPT" and true_var_name = "TrueDeltaPT"), in which case true_var_name is optional.
        preselection : str
            The name of one of the sets of preselection cuts in selections.py in microfit.
        selection : str
            The name of one of the sets of selection cuts in selections.py in microfit.
        truth_def : str
            A set of cuts specifying the signal definition using truth-level variables.
        n_bins : int, optional
            Number of bins used in the distribution. If not provided, then bin_edges must be provided.
        x_range : tuple, optional
            Tuple of lower and upper limits. If not provided, then bin_edges must be provided.
        bin_edges : np.ndarray, optional
            Array of bin edges. If this is provided, the n_bins and limits are ignored.
        weightCV : str
            The central value weights for the CV histogram. In the PELEE framework, the column "weights" = "weightSplineTimesTune" * data_pot / mc_pot
            so additional scaling is not needed.
        base_weight_Var : str
            Name of the column containing the baseline weights of the events to be used for the variation histograms
            (as this function is to calculate the GENIE multisim, we need to use the weights without the GENIE tune). 
            In the PELEE framework, the column "weights_no_tune" = "weightSpline" * data_pot / mc_pot, so additional scaling is not needed.
        wname : str
            The name of the column containing the multisim weights of the events.

        Returns
        -------
        covariance_matrix : array_like
            Covariance matrix of the bin counts.
        """
        
        # For my own 1e1p TKI branch, I have named my variables in a certain way - not applicable in general
        if true_var_name is None:
                label = var_name.lstrip("Reco")
                true_var_name = "True" + label

        # Sometimes n_bins is none in the Binning object if using variable bin sizes
        if bin_edges is not None:
            n_bins = len(bin_edges) - 1
            bins = bin_edges
        elif x_range is None:
            bins = n_bins
        else:
            bins = np.linspace(x_range[0],x_range[1],n_bins+1)

        Nuniverse = len(rundata["mc"][wname][0]) # should be 100 in the 2024 PELEE n-tuples

        n_tot = np.zeros([Nuniverse, n_bins]) # this will store the variation hists (rows)
        n_cv_tot = np.zeros(n_bins) # this will store the sum of the CV hists

        # First calculate the covariance for the background
        for key, df in rundata.items():
            if key not in ["data", "ext"]: # the calculation should exclude all data and only be performed on MC
                #
                extra_query = ""
                if key == signame:
                    extra_query = f"& ~({truth_def})" # removing the signal events
                        
                from microfit import selections as sel
                query = f"{sel.preselection_categories[preselection]['query']} and {sel.selection_categories[selection]['query']}"

                queried_df = df.query(query+extra_query, engine="python")
                variable = queried_df[var_name]
                CV_base_weight = queried_df[weightCV]
                Var_base_weight = queried_df[base_weight_Var]
                syst_weights = queried_df[wname]

                # Calculate the background CV hist sum
                n_cv, bins = np.histogram(variable, bins=bins, weights=CV_base_weight)
                n_cv_tot += n_cv

                # Calculate the background variation hists and store them

                weights_df = pd.DataFrame(syst_weights.values.tolist()) # df of multisim weights that have been flattened horizontally - i.e. Nuniv (100) columns per row

                if not weights_df.empty:
                    for i in range(Nuniverse):
                        weight = weights_df[i].values / 1000.
                        weight[np.isnan(weight)] = 1
                        weight[weight > 100] = 1
                        weight[weight < 0] = 1
                        weight[weight == np.inf] = 1

                        n, bins = np.histogram(variable, weights=weight * Var_base_weight,bins=bins)
                        n_tot[i] += n

        # Now calculate the covariance for the signal using the response matrix method

        df = rundata[signame]
        queried_df = df.query(f"{query} & ({truth_def})", engine="python")
        variable = queried_df[var_name]
        CV_base_weight = queried_df[weightCV]
        Var_base_weight = queried_df[base_weight_Var]
        syst_weights = queried_df[wname]

        # Calculate the signal CV hist sum
        n_cv, bins = np.histogram(variable, bins=bins, weights=CV_base_weight)
        n_cv_tot += n_cv

        # Calculate the signal variation hists and store them

        weights_df = pd.DataFrame(syst_weights.values.tolist()) # df of multisim weights that have been flattened horizontally - i.e. Nuniv (100) columns per row

        # Calculate the true signal CV that stays constant in all the universes
        true_sig = df.query(truth_def, engine="python")
        true_variable = true_sig[true_var_name]
        true_var_weightsCV  = true_sig[weightCV]
        t_cv, bins = np.histogram(true_variable, bins=bins, weights=true_var_weightsCV)

        if not weights_df.empty:
            for i in range(Nuniverse):
                rmv, xb, yb = ResponseMatrix(df, truth_def, query, true_var_name, var_name, bins, base_weight_Var, i, wname)
                #print(rmv)
                rp = rmv.dot(t_cv)
                #print("variation: ",rp)
                n_tot[i] += rp

        # Now finally compute the covariance

        cov = np.zeros([len(n_cv_tot), len(n_cv_tot)])
        for n in n_tot:
                for i in range(len(n_cv_tot)):
                        for j in range(len(n_cv_tot)):
                                cov[i][j] += (n[i] - n_cv_tot[i]) * (n[j] - n_cv_tot[j])
        
        cov /= Nuniverse

        return cov

In [35]:
def unisim_err_with_resp_func(rundata, signame, var_name, true_var_name, preselection, selection, truth_def, n_bins=None, x_range=None, bin_edges=None, weightCV="weights", base_weight_Var="weights_no_tune"):

    """Calculate unisim uncertainties using the response matrix method.

        Unisim means that a single variation of a given analysis input parameter is performed according to its uncertainty.
        The difference in the number of selected events between this variation and the central value is taken as the
        uncertainty in that number of events. Mathematically, this is the same as the 'multisim' method, but with only
        one or two universes.

        This particular method, whereby the response matrix is used, is required for xsec analyses.
        Here, the signal and background are treated differently. The covariance for the background
        is calculated as described above. But the covariance for the signal is calculated using the
        response matrix; the variation for a particular universe is calculated by multiplying the
        response matrix for that universe (S^{univ}_{sel} / S^{univ}_{total}) by the total true
        signal CV (central value) prediction.
        
        This function has written to be used with microfit.

        Parameters
        ----------
        rundata : dict
            Dictionary of dataframes for each sample. This is the output from the load_runs()
            function in data_loading.py.
        signame : str
            The name of the MC sample that contains your signal events. This must refer to one of the keys in rundata.
            For my analysis, this is "nue".
        var_name : str
            The name of the reconstructed variable you are calculating the covariance for.
        true_var_name : str
            The name of the truth-level variable corresponding to var_name. This is required, unless var_name and true_var_name
            share the same stem but with either "Reco" or "True" at the front of the string 
            (e.g. var_name = "RecoDeltaPT" and true_var_name = "TrueDeltaPT"), in which case true_var_name is optional.
        preselection : str
            The name of one of the sets of preselection cuts in selections.py in microfit.
        selection : str
            The name of one of the sets of selection cuts in selections.py in microfit.
        truth_def : str
            A set of cuts specifying the signal definition using truth-level variables.
        n_bins : int, optional
            Number of bins used in the distribution. If not provided, then bin_edges must be provided.
        x_range : tuple, optional
            Tuple of lower and upper limits. If not provided, then bin_edges must be provided.
        bin_edges : np.ndarray, optional
            Array of bin edges. If this is provided, the n_bins and limits are ignored.
        weightCV : str
            The central value weights for the CV histogram. In the PELEE framework, the column "weights" = "weightSplineTimesTune" * data_pot / mc_pot
            so additional scaling is not needed.
        base_weight_Var : str
            Name of the column containing the baseline weights of the events to be used for the variation histograms
            (as this function is to calculate the GENIE multisim, we need to use the weights without the GENIE tune). 
            In the PELEE framework, the column "weights_no_tune" = "weightSpline" * data_pot / mc_pot, so additional scaling is not needed.

        Returns
        -------
        cov : array_like
            Covariance matrix of the bin counts.
    """

    # For my own 1e1p TKI branch, I have named my variables in a certain way - not applicable in general
    if true_var_name is None:
        label = var_name.lstrip("Reco")
        true_var_name = "True" + label

    # Sometimes n_bins is none in the Binning object if using variable bin sizes
    if bin_edges is not None:
        n_bins = len(bin_edges) - 1
        bins = bin_edges
    elif x_range is None:
        bins = n_bins
    else:
        bins = np.linspace(x_range[0],x_range[1],n_bins+1)

    knob_v = ['knobRPA','knobCCMEC','knobAxFFCCQE','knobVecFFCCQE','knobDecayAngMEC','knobThetaDelta2Npi']
    knob_n = [2,1,1,1,1,1]

    n_cv_tot = np.zeros(n_bins) # this will store the sum of the CV hists
    n_tot_v = [] # this will store the variation hists
    for u, knob in enumerate(knob_v):
        n_tot_v.append(np.zeros([ knob_n[u] ,n_bins]))

    # First calculate the covariance for the background
    for key, df in rundata.items():
        if key not in ["data", "ext"]: # the calculation should exclude all data and only be performed on MC
            extra_query = ""
            if key == signame:
                extra_query = f"& ~({truth_def})" # removing the signal events
                        
            from microfit import selections as sel
            query = f"{sel.preselection_categories[preselection]['query']} and {sel.selection_categories[selection]['query']}"

            queried_df = df.query(query+extra_query, engine="python")
            variable = queried_df[var_name]
            CV_base_weight = queried_df[weightCV]
            Var_base_weight = queried_df[base_weight_Var]
            #syst_weights = queried_df[wname]

            # Calculate the background CV hist sum
            n_cv, bins = np.histogram(variable, bins=bins, weights=CV_base_weight)
            n_cv_tot += n_cv

            # Calculate the background variation hists and store them

            for n, knob in enumerate(knob_v):
                
                weight_up = queried_df[f"{knob}up"].values
                weight_up[np.isnan(weight_up)] = 1
                weight_up[weight_up > 100] = 1
                weight_up[weight_up < 0] = 1
                weight_up[weight_up == np.inf] = 1
                n_up, bins = np.histogram(variable, weights=weight_up * Var_base_weight, bins=bins)
                n_tot_v[n][0] += n_up

                if (knob_n[n] == 2):
                    weight_dn = queried_df[f"{knob}dn"].values
                    weight_dn[np.isnan(weight_dn)] = 1
                    weight_dn[weight_dn > 100] = 1
                    weight_dn[weight_dn < 0] = 1
                    weight_dn[weight_dn == np.inf] = 1
                    n_dn, bins = np.histogram(variable, weights=weight_dn * Var_base_weight, bins=bins)
                    n_tot_v[n][1] += n_dn

    # Now calculate the covariance for the signal using the response matrix method

    df = rundata[signame]
    queried_df = df.query(f"{query} & ({truth_def})", engine="python")
    variable = queried_df[var_name]
    CV_base_weight = queried_df[weightCV]
    Var_base_weight = queried_df[base_weight_Var]

    # Calculate the signal CV hist sum
    n_cv, bins = np.histogram(variable, bins=bins, weights=CV_base_weight)
    n_cv_tot += n_cv

    # Calculate the signal variation hists and store them

    # Calculate the true signal CV that stays constant in all the universes
    true_sig = df.query(truth_def, engine="python")
    true_variable = true_sig[true_var_name]
    true_var_weightsCV  = true_sig[weightCV]
    t_cv, bins = np.histogram(true_variable, bins=bins, weights=true_var_weightsCV)

    for n,knob in enumerate(knob_v):
        
        rmv_up, xb, yb = ResponseMatrix(df, truth_def, query ,true_var_name, var_name, bins, base_weight_Var, 0, f"{knob}up")
        rp_up = rmv_up.dot(t_cv)
        n_tot_v[n][0] += rp_up
        if (knob_n[n] == 2):
            rmv_dn, xb, yb = ResponseMatrix(df,truth_def, query, true_var_name, var_name, bins, base_weight_Var, 0, f"{knob}dn")
            rp_dn = rmv_dn.dot(t_cv)
            n_tot_v[n][1] += rp_dn

     # Now finally compute the covariance
    
    cov = np.zeros([len(n_cv_tot), len(n_cv_tot)])

    for n,knob in enumerate(knob_v):
        
        this_cov = np.zeros([len(n_cv_tot), len(n_cv_tot)])

        if (knob_n[n] == 2):
            for i in range(len(n_cv)):
                #print ('knob %s has CV: %.0f, VAR UP: %.0f, VAR DN: %.0f entries'%(knob,n_cv_tot[i],n_tot_v[n][0][i],n_tot_v[n][1][i]))
                for j in range(len(n_cv)):
                    this_cov[i][j] += (n_tot_v[n][0][i] - n_cv_tot[i]) * (n_tot_v[n][0][j] - n_cv_tot[j])
                    this_cov[i][j] += (n_tot_v[n][1][i] - n_cv_tot[i]) * (n_tot_v[n][1][j] - n_cv_tot[j])
            this_cov /= 2.

        if (knob_n[n] == 1):
            for i in range(len(n_cv)):
                #print ('knob %s has CV: %.0f, VAR: %.0f entries'%(knob,n_cv_tot[i],n_tot_v[n][0][i]))
                for j in range(len(n_cv)):
                    this_cov[i][j] += (n_tot_v[n][0][i] - n_cv_tot[i]) * (n_tot_v[n][0][j] - n_cv_tot[j])

        cov += this_cov

        return cov

In [25]:
def ResponseMatrix(df, truth_def, fullsel, var_name, true_var_name, bin_edges, base_weight_Var, univ=-1, wname=""):

    """Calculate the response matrix.

        Parameters
        ----------
        df : pandas dataframe
            The dataframe of the n-tuple being used to calculate the response matrix.
        truth_def : str
            A set of cuts specifying the signal definition using truth-level variables.
        var_name : str
            The name of the reconstructed variable you are calculating the covariance for.
        true_var_name : str
            The name of the truth-level variable corresponding to var_name. 
        bin_edges : np.ndarray
            Array of bin edges.
        base_weight_Var : str
            Name of the column containing the baseline weights of the events. 

        Returns
        -------
        rm : array_like
            Response matrix of the bin counts.
        xb : array_like
            Horizontal bin edges.
        yb : array_like
            Vertical bin edges.
    """
    
    # Get number of signal events at true level before selection
    true_sig = df.query(truth_def, engine="python")
    truevals = true_sig[true_var_name]
    tweights = true_sig[base_weight_Var]
    if univ>=0:
        syst_weights = true_sig[wname]
        vweights_df = pd.DataFrame(syst_weights.values.tolist()) # df of multisim weights that have been flattened horizontally - i.e. Nuniv (100) columns per row
        vweights = vweights_df[univ].values / 1000.
        vweights[np.isnan(vweights)] = 1
        vweights[vweights > 100] = 1
        vweights[vweights < 0] = 1
        vweights[vweights == np.inf] = 1
        tweights = tweights * vweights
    n, bins = np.histogram(truevals, weights=tweights, bins=bin_edges)

    # Get number of signal events at reco and true level after selection
    sel_true_sig = true_sig.query(fullsel, engine="python")
    x = sel_true_sig[true_var_name]
    y = sel_true_sig[var_name]
    w = sel_true_sig[base_weight_Var]
    if univ>=0:
        sw = sel_true_sig[wname]
        vw_df = pd.DataFrame(sw.values.tolist()) # df of multisim weights that have been flattened horizontally - i.e. Nuniv (100) columns per row
        vw = vw_df[univ].values / 1000.
        vw[np.isnan(vw)] = 1
        vw[vw > 100] = 1
        vw[vw < 0] = 1
        vw[vw == np.inf] = 1
        w = w * vw
    H, xb, yb = np.histogram2d(x, y, weights=w, bins=[bin_edges, bin_edges])
        
    # Get response matrix
    rm = np.transpose(H)/n
    return rm, xb, yb

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append("../../../")
import data_loading as dl
from importlib import reload
reload(dl)

from microfit import run_plotter as rp
from microfit import histogram as hist

from microfit import variable_definitions as vdef
from microfit import selections

from microfit import xsec_covariances as xs

In [2]:
keep_vars = [
    "Signal_1e1p", "mc_signal_1e1p", "nu_pdg", "TrueElecIdx", "TrueLeadProtonIdx", "InFV", "HasNoMesons",
    "TrueNElec", "TrueNProt", "TrueDeltaPT", "TrueDeltaAlphaT", "TruePN", "TrueAlpha3D",
    "nproton", "npion", "npi0", "nelec", "nmuon", "isVtxInFiducial",
    "Sel_1e1p", "sel_1e1p_w_cuts", "RecoElectronCandidateIdx", "RecoLeadProtonCandidateIdx", "InFV_reco",
    "RecoElecPassMomCut", "RecoLeadProtonPassMomCut", "n_reco_tracks", "n_reco_showers",
    "RecoDeltaPT", "RecoDeltaAlphaT", "RecoPN", "RecoAlpha3D", "RecoECal", "Reco_mag_q", "RecoPL",
    "nslice", "selected", "shr_energy_tot_cali", "_opfilter_pe_beam", "_opfilter_pe_veto", "bnbdata", "extdata",
    "CosmicIPAll3D", "hits_ratio", "shrmoliereavg", "subcluster", "trkfit", "trkshrhitdist2", "tksh_distance",
    "shr_tkfit_nhits_tot", "shr_tkfit_dedx_max", "tksh_angle", "shr_trk_len", "reco_e",
    "RecoLeadProton_trk_len", "RecoLeadProton_trk_trunk_dEdx_y", "RecoLeadProton_dEdx_y_per_trklen",
    "RecoLeadProtonCandidate_trk_pid", "RecoElectronCandidate_shr_pid", "RecoElectron_conversion_dist",
    "pi0_radlen1", "pi0_radlen2", "pi0_score", "nonpi0_score", "bkg_score",
    "RecoElecE", "RecoElecModMom", "RecoElecMomX", "RecoElecMomY", "RecoElecMomZ",
    "RecoLeadProtonKE", "RecoLeadProtonModMom", "RecoLeadProtonMomX", "RecoLeadProtonMomY", "RecoLeadProtonMomZ",
]

In [3]:
RUN = ["3"]
#RUN = ["1","2","3","4a","4b","4c","4d","5","1A_OT","1B_OT"] # for detvars with bnb or for closure test
#RUN = ["1","2","3","4c","5"] # for nuwro_fd, no run 4b and 4d available
blinded = False
#data="nuwro_fd"
data="bnb"

In [4]:
rundata, mc_weights, data_pot = dl.load_runs(
    RUN,
    data=data,
    loadpi0variables=False,
    loadshowervariables=True,
    loadrecoveryvars=False,
    loadsystematics=True,
    numupresel=False,
    loadnumuvariables=False,
    use_bdt=True,
    load_lee=False,
    load_nue_tki=True,
    #keep_columns=keep_vars,
    blinded=blinded,
    load_crt_vars=False,
    enable_cache=True,
)

Loading run 3


In [5]:
run_combo = "Run"
for run in RUN:
    run_combo += run

In [6]:
selection = "OnePBDT"
preselection = "OneP_new"
genie_multisim_cov = xs.multisim_err_with_resp_func(
    rundata, 
    "nue", 
    "RecoDeltaPT", 
    "TrueDeltaPT", 
    preselection, 
    selection, 
    "category_1e1p == 12",  
    bin_edges = [0,0.3,1.7],
)

print("multisim: \n", genie_multisim_cov)

multisim: 
 [[112.08385982   6.75620598]
 [  6.75620598   1.26779438]]


In [7]:
genie_unisim_cov = xs.unisim_err_with_resp_func(
    rundata, 
    "nue", 
    "RecoDeltaPT", 
    "TrueDeltaPT", 
    preselection, 
    selection, 
    "category_1e1p == 12",  
    bin_edges = [0,0.3,1.7],
)

print("uniisim: \n", genie_unisim_cov)

uniisim: 
 [[108.66762016  21.91504826]
 [ 21.91504826   4.42603258]]


In [8]:
for binning_def in vdef.TKI_variables_1e1p:
    binning = hist.Binning.from_config(*binning_def)
    print(binning)
    print()

Binning(variable='RecoDeltaPT', bin_edges=array([0. , 0.3, 1.7]), label='RecoDeltaPT', variable_tex='$\\delta p_T$ [GeV/c]', variable_tex_short=None, is_log=False, selection_query=None, selection_key=None, preselection_key=None, selection_tex=None, selection_tex_short=None)

Binning(variable='RecoDeltaAlphaT', bin_edges=array([  0.,  80., 180.]), label='RecoDeltaAlphaT', variable_tex='$\\delta \\alpha_T$ [degrees]', variable_tex_short=None, is_log=False, selection_query=None, selection_key=None, preselection_key=None, selection_tex=None, selection_tex_short=None)

Binning(variable='RecoPN', bin_edges=array([0. , 0.3, 1.7]), label='RecoPN', variable_tex='$p_n$ [GeV/c]', variable_tex_short=None, is_log=False, selection_query=None, selection_key=None, preselection_key=None, selection_tex=None, selection_tex_short=None)

Binning(variable='RecoAlpha3D', bin_edges=array([  0.,  90., 180.]), label='RecoAlpha3D', variable_tex='$\\alpha_{3D}$ [degrees]', variable_tex_short=None, is_log=False, s

In [18]:
print(binning.variable)
print(type(binning.bin_edges))
print(binning_def[-1])
print(type(binning_def[-1]))

RecoAlpha3D
<class 'numpy.ndarray'>
[0, 90.0, 180.0]
<class 'list'>


In [19]:
genie_cov = xs.multisim_err_with_resp_func(
    rundata, 
    "nue", 
    binning.variable, 
    None, 
    preselection, 
    selection, 
    "category_1e1p == 12",  
    bin_edges = binning.bin_edges,
)
print(genie_cov)

[[ 2.25616941 11.68740354]
 [11.68740354 65.86265905]]


In [20]:
genie_unisim_cov = xs.unisim_err_with_resp_func(
    rundata, 
    "nue", 
    binning.variable, 
    None, 
    preselection, 
    selection, 
    "category_1e1p == 12",  
    bin_edges = binning.bin_edges,
)
print(genie_unisim_cov)

[[6.42738224e-02 2.34393725e+00]
 [2.34393725e+00 9.79605312e+01]]


In [ ]:
print(mc_weights)

{'data': 1.0, 'ext': 0.2962600445651671, 'mc': 0.19603053435114504, 'nue': 0.0033050193050193047, 'drt': 0.7853211009174312}


In [18]:
print("weights_no_tune" in rundata["mc"].columns)
print(len(rundata["mc"]["weightsGenie"][0]))
rundata["nue"].loc[:, ("weightsGenie", "weightSpline", "weights", "weightSplineTimesTune", "weights_no_tune")]

True
100


,weightsGenie,weightSpline,weights,weightSplineTimesTune,weights_no_tune
entry,,,,,
0,"[1914, 405, 1060, 730, 505, 1791, 1066, 829, 8...",0.981016,0.003779,1.143481,0.003242
1,"[1295, 865, 746, 1131, 1996, 1225, 1211, 1091,...",1.038154,0.004095,1.239054,0.003431
2,"[2081, 892, 641, 1719, 1549, 1732, 1722, 1235,...",0.974409,0.004102,1.241263,0.003220
3,"[1195, 927, 769, 1116, 1683, 1192, 1162, 1152,...",1.648609,0.006388,1.932914,0.005449
4,"[1209, 986, 863, 1131, 1634, 1220, 1167, 1169,...",0.965846,0.003852,1.165619,0.003192
...,...,...,...,...,...
61952,"[1269, 1180, 915, 1295, 730, 1442, 1061, 1255,...",0.964037,0.003737,1.130834,0.003186
61953,"[980, 424, 542, 1388, 948, 550, 1293, 494, 142...",0.965515,0.003643,1.102224,0.003191
61954,"[1499, 348, 1308, 543, 481, 1269, 901, 596, 62...",0.974858,0.003107,0.939972,0.003222


In [ ]:
s = rundata["mc"]["weightsGenie"]
df = pd.DataFrame(s.values.tolist())
print(df[1].values)
df.head(20)

[1379 3492 1127 ...  660 1183  384]


,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,985,1379,1344,719,452,1021,574,838,1099,400,...,1325,942,875,1001,625,940,820,1062,953,1241
1,832,3492,1563,764,381,622,1221,813,1818,152,...,2262,1626,113,914,296,1441,1046,877,1084,5
2,1374,1127,789,888,682,1186,968,1402,824,958,...,504,765,992,1612,1336,963,1065,662,1097,759
3,699,1059,484,1285,1847,1139,1370,1357,558,1428,...,1501,424,1683,1611,763,653,1679,1192,905,1279
4,335,308,182,2007,3478,374,1935,433,977,717,...,796,1398,1694,641,729,0,682,873,1054,1040
5,1201,1283,1308,571,1303,763,646,611,889,386,...,1371,659,1308,419,868,1002,521,760,709,1365
6,334,1947,472,600,1169,3265,1410,883,187,1981,...,8395,498,2867,1322,176,1009,1149,208,2728,1491
7,782,782,929,1034,489,907,787,494,661,1744,...,580,974,758,858,982,951,339,1616,841,1312
8,1208,1200,1007,907,704,1038,940,1038,875,995,...,665,751,1112,934,987,1062,877,862,936,888
9,1009,1315,1584,696,1013,831,668,716,1136,371,...,1681,1129,1043,772,623,995,884,889,813,1254


In [ ]:
print(np.stack(s).ndim>1)
if (np.stack(s).ndim>1):
    #multisim, pick specific universe
    p = np.stack(s)[:,0]/1 #000.
    print(p)
print(p)

False


NameError: name 'p' is not defined

In [ ]:
rundata["mc"].loc[:, ("weights_no_tune")]

entry
0        0.196031
1        0.196031
2        0.196031
3        0.196031
4        0.196031
           ...   
32181    0.196031
32182    0.196031
32183    0.196031
32184    0.196031
32185    0.196031
Name: weights_no_tune, Length: 31075, dtype: float64